# 10 — When Is the 24h-Lag ("Climatology") a Fair Reference?

**Question.** The 24h-lag baseline is not uniformly weak or strong: it should win
where the diurnal cycle dominates and the forecast window spans a diurnal
transition. If so, "skill vs climatology" means different things at different
stations and hours, and one global number hides that.

> ### Binning pitfall — read before changing anything here
> A first version of this analysis binned by **civil local time** using the
> evaluation's **90-minute origin stride**. That is broken, and visibly so:
> targets then land on only **16 distinct UTC clock times**, leaving 8 UTC hour
> bins empty; converting to Europe/Zurich smears DST across them, so local-hour
> bins end up with alternating seasonal composition —
> `hour 0: 0% summer, hour 1: 64%, hour 2: 97%, hour 3: 0%, ...`
> The "hour of day" axis was substantially a **season** axis, and the giveaway
> was a 3-hour sawtooth in **pressure**, which has no diurnal cycle at that
> amplitude.
>
> **Fixes applied below, both required:**
> 1. **stride-1 origins** for the baselines (they need no model, so every
>    10-min step can be an origin) → all 144 clock times sampled equally.
> 2. **bin by UTC**, never civil time. Swiss solar time ≈ UTC + 0:32, so UTC
>    tracks solar position while avoiding DST entirely.
>
> Model curves are stuck with the dump's 16 origin times, so §2 bins the model
> by UTC too and shows only the populated bins.

**Findings after the fix** (raw parquet, 105,372 origins; figure
`10_clim_vs_pers_structure.png`, table `10_clim_win_fraction_by_station.csv`):

| lead | variable | pers MAE | 24h-lag MAE | cells 24h-lag wins | ρ(altitude) |
|---|---|---|---|---|---|
| +2h | temperature | 1.261 | 2.451 | 5.3% | −0.35 |
| +2h | pressure | 0.561 | 3.464 | 0.0% | −0.09 |
| +6h | temperature | 2.934 | 2.450 | **51.3%** | **−0.67** |
| +6h | humidity | 13.199 | 12.724 | **45.2%** | **−0.63** |
| +6h | pressure | 1.347 | 3.463 | **0.0%** | −0.09 |
| +6h | wind_u / wind_v | 1.617 / 1.566 | 1.861 / 1.785 | 28% / 32% | −0.27 / −0.34 |

* **Lead-dependent.** At +2h the 24h-lag is hopeless (≈5% of cells); by +6h it
  wins the majority for temperature.
* **Hour-dependent, smooth single cycle.** Temperature +6h: ratio bottoms at
  **0.52 at 11 UTC** (targets around local midday — the window spans morning
  warm-up, where persistence fails worst) and peaks at **1.74 at 05 UTC**
  (stable pre-dawn, persistence strong).
* **Altitude-dependent.** ρ = −0.67 (temperature), −0.63 (humidity). Valley
  stations with strong radiative diurnal cycles — MAG (203 m), SIO (482 m),
  VIS (639 m), ILZ (698 m) — the 24h-lag wins ~80% of hours. High alpine —
  GOR (3129 m), DIA (2964 m), ATT (2734 m), GRH, MLS — it wins **0%**.
* **Pressure never**, at any lead, hour or altitude (ρ ≈ −0.09): synoptic
  autocorrelation beats the diurnal cycle everywhere.

**This notebook** adds the models: does v27 beat the 24h-lag precisely in the
hours and stations where persistence collapses?

In [ ]:
# ── Bootstrap ────────────────────────────────────────────────────────────────
import os, sys
import numpy as np, pandas as pd, matplotlib.pyplot as plt
for _cand in (os.getcwd(),
              os.path.join(os.getcwd(), "notebooks", "analysis"),
              os.path.dirname(os.path.abspath("__file__"))):
    if os.path.isfile(os.path.join(_cand, "common.py")):
        if _cand not in sys.path: sys.path.insert(0, _cand)
        break
import importlib
import common as C
# common.py is edited while these notebooks are open. A plain import is cached
# by the kernel, so edits are invisible until a restart — and a stale module
# silently serves stale caches (this is exactly how a station_table without
# easting/northing survived a fix to station_table). Reload every run.
importlib.reload(C)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
RUNS = C.discovered_runs()
for r, d in RUNS.items(): print(f"{r:20s} {sorted(d)}")

## 1. Baseline structure by local hour, from `raw_test_obs` (verified path)

In [ ]:
raw = C.raw_test_obs(); ns = C.norm_stats(); stn = C.station_table()
VARS = ns["var_names"]; OBS, OM = raw["obs"], raw["mask"]
NT = raw["nt"]
SEASONS = ["DJF", "MAM", "JJA", "SON"]

# Absolute time of every row, built once and vectorised.
# Bin by UTC, NOT civil local time — DST would alias the bins against the
# origin stride (see the warning at the top). Swiss solar time is UTC + ~0:32.
IDX = (pd.Timestamp("1970-01-01", tz="UTC")
       + pd.to_timedelta(raw["h0"] + np.arange(NT) / 6.0, unit="h"))
utc_h = IDX.hour.values.astype(np.int8)
_m = IDX.month.values
sea_row = np.select([np.isin(_m, [12, 1, 2]), np.isin(_m, [3, 4, 5]),
                     np.isin(_m, [6, 7, 8])], [0, 1, 2], 3).astype(np.int8)

# stride-1 origins: baselines need no model, so every step can be an origin.
# This is what makes each (season, UTC hour) bin equally populated.
orig = np.arange(C.CLIM_LAG_STEPS, NT - 36, 1)
res = {}                       # (season, hour, station, variable)
for lab, h in {"+2h": 12, "+6h": 36}.items():
    sp = np.zeros((4, 24, len(stn), 5)); sc = np.zeros_like(sp)
    cn = np.zeros_like(sp)
    for a in range(0, len(orig), 20000):
        o = orig[a:a + 20000]; tr = o + h
        tgt, per, cli = OBS[tr], OBS[o], OBS[tr - C.CLIM_LAG_STEPS]
        ok = OM[tr] & OM[o] & OM[tr - C.CLIM_LAG_STEPS]
        # season and hour of the TARGET time
        np.add.at(sp, (sea_row[tr], utc_h[tr]), np.where(ok, np.abs(per - tgt), 0))
        np.add.at(sc, (sea_row[tr], utc_h[tr]), np.where(ok, np.abs(cli - tgt), 0))
        np.add.at(cn, (sea_row[tr], utc_h[tr]), ok.astype(float))
    res[lab] = (sp, sc, cn)
    n = cn[:, :, :, 0].sum(2)
    print(f"{lab}: per (season,hour) bin counts "
          f"min {n.min()/1e3:.0f}k max {n.max()/1e3:.0f}k")
# all-season view, for the aggregate panels further down
res_all = {k: tuple(x.sum(0) for x in v) for k, v in res.items()}
assert res["+6h"][2][:, :, :, 0].sum(2).min() > 0, "empty (season, hour) bin"

## 2. Model error by UTC hour of target (+2h), split by season

Rows are seasons, columns are variables. Each panel carries persistence, the
24h-lag, and the models, all binned by UTC hour of the target.

The seasonal split matters because the diurnal cycle itself is seasonal: the
day-night temperature swing is far larger in JJA than DJF, so persistence's
twin failure peaks (morning warm-up, evening cooling) should be much taller in
summer — and if the models have genuinely learned diurnal physics rather than
season-averaged behaviour, their curves should stay comparatively flat in
*every* season, not just on annual average.

Model curves appear only on the 16 populated UTC hours (the evaluation origins
sit on a 90-minute grid); baselines use all 24.

In [ ]:
üW = {r: C.window_errors(r, "mr0.00", leads=(1, 4))
     for r in ["v27", "lstm-baseline-v1"]}
STD = ns["std"]; li = 1                                   # +2h

# season + UTC hour of each window's TARGET (origin + 2 h)
TGT = (pd.Timestamp("1970-01-01", tz="UTC")
       + pd.to_timedelta(W["v27"]["t0_hours"] + 2.0, unit="h"))
hh_t = TGT.hour.values.astype(np.int8)
_mt = TGT.month.values
sea_t = np.select([np.isin(_mt, [12, 1, 2]), np.isin(_mt, [3, 4, 5]),
                   np.isin(_mt, [6, 7, 8])], [0, 1, 2], 3).astype(np.int8)
FILLED = np.array(sorted(set(hh_t.tolist())))
print("populated UTC hours:", FILLED.tolist())
print("windows per season:", {s: int((sea_t == i).sum())
                              for i, s in enumerate(SEASONS)})

sp, sc, cn = res["+2h"]
# sharey="col": one y-scale per VARIABLE across all four seasons. Without it
# each panel auto-scales and a flat JJA panel may sit an octave above a flat
# DJF one — the seasonal comparison the figure exists for becomes unreadable.
fig, axes = plt.subplots(4, 5, figsize=(18, 11), sharex=True, sharey="col")
_lim = {vi: [np.inf, -np.inf] for vi in range(len(VARS))}
for si, seas in enumerate(SEASONS):
    for vi, v in enumerate(VARS):
        ax = axes[si, vi]
        c_ = np.maximum(cn[si, :, :, vi].sum(1), 1)
        ax.plot(range(24), sp[si, :, :, vi].sum(1) / c_, "--",
                color="#888", lw=1.3, label=C.BASELINE_LABELS["persistence"])
        ax.plot(range(24), sc[si, :, :, vi].sum(1) / c_, ":",
                color="#B08968", lw=1.8, label=C.BASELINE_LABELS["clim"])
        for r in W:
            e = W[r]["err_norm"][li, :, :, vi].astype(np.float32) * STD[None, :, vi]
            ok = W[r]["valid"][li, :, :, vi]
            y = []
            for h in FILLED:
                sel = (hh_t == h) & (sea_t == si)
                y.append(np.nanmean(np.where(ok[sel], e[sel], np.nan))
                         if sel.any() else np.nan)
            ax.plot(FILLED, y, "o-", ms=2.5, lw=1.2,
                    color=C.MODELS[r][1], label=C.MODELS[r][0])
            _yy = np.asarray(y, dtype=float)
            if np.isfinite(_yy).any():
                _lim[vi][0] = min(_lim[vi][0], np.nanmin(_yy))
                _lim[vi][1] = max(_lim[vi][1], np.nanmax(_yy))
        _p = sp[si, :, :, vi].sum(1) / c_                 # persistence fits
        if np.isfinite(_p).any():
            _lim[vi][1] = max(_lim[vi][1], np.nanmax(_p))
        ax.grid(alpha=.3); ax.set_xticks(range(0, 24, 6))
        if si == 0: ax.set_title(f"{v}  [{C.UNITS[v]}]", fontsize=10)
        if vi == 0: ax.set_ylabel(f"{seas}\nMAE [phys]", fontsize=9)
        if si == 3: ax.set_xlabel("UTC hour of target")
# One scale per variable. Limits come from the models and persistence; the
# 24h-lag is ~2.6x the model for humidity and would flatten everything else.
for vi in range(len(VARS)):
    lo, hi = _lim[vi]
    if np.isfinite(lo) and np.isfinite(hi) and hi > lo:
        pad = 0.05 * (hi - lo)
        for si in range(4):
            axes[si, vi].set_ylim(lo - pad, hi + pad)
axes[0, -1].legend(fontsize=6.5)
fig.suptitle("+2h error by UTC hour of target, split by season"
             "  ·  y-scale shared per variable (24h-lag may be clipped)",
             y=1.005)
plt.tight_layout(); C.save_fig(fig, "10_model_vs_baselines_hod_season"); plt.show()

In [ ]:
# ── Diurnal amplitude per season: how much of persistence's swing survives? ──
# persistence fails at the diurnal TRANSITIONS. A model that has learned them
# has a flatter curve. Amplitude = max/min over the populated UTC hours.
def hod_curve(run, vi, season=None):
    e = W[run]["err_norm"][li, :, :, vi].astype(np.float32) * STD[None, :, vi]
    ok = W[run]["valid"][li, :, :, vi]
    out = np.full(24, np.nan)
    for h in FILLED:
        sel = (hh_t == h) if season is None else ((hh_t == h) & (sea_t == season))
        if sel.any():
            out[h] = np.nanmean(np.where(ok[sel], e[sel], np.nan))
    return out

amp = lambda a: np.nanmax(a[FILLED]) / np.nanmin(a[FILLED])
rows = []
for si, seas in enumerate(SEASONS + ["ALL"]):
    s_ = None if seas == "ALL" else si
    P_ = (sp.sum(0) if s_ is None else sp[s_])
    C_ = (cn.sum(0) if s_ is None else cn[s_])
    for vi, v in enumerate(VARS):
        P = P_[:, :, vi].sum(1) / np.maximum(C_[:, :, vi].sum(1), 1)
        rec = {"season": seas, "variable": v, "persistence": amp(P)}
        for r in W:
            rec[C.MODELS[r][0]] = amp(hod_curve(r, vi, s_))
        M = hod_curve("v27", vi, s_)
        pk = FILLED[int(np.nanargmax(P[FILLED]))]
        mn = FILLED[int(np.nanargmin(P[FILLED]))]
        rec["skill@pers-peak"] = 1 - M[pk] / P[pk]
        rec["skill@pers-min"] = 1 - M[mn] / P[mn]
        rows.append(rec)
t = pd.DataFrame(rows).set_index(["season", "variable"])
display(t.round(3).style.background_gradient(cmap="viridis_r",
                                             subset=["persistence"]))
C.save_table(t, "10_diurnal_amplitude_by_season")
print("persistence amplitude should be largest in JJA (strongest diurnal cycle);")
print("if the model amplitudes stay near 1 in EVERY season, the diurnal cycle")
print("is genuinely learned rather than absorbed as a seasonal average.")

## Interpretation

**Sanity check first.** If any baseline curve shows a repeating 2-3 hour
sawtooth, the binning has regressed (see the warning at the top). Curves should
be smooth. Model curves appear only on the 16 populated UTC hours.

**Established (annual, +2h, physical units).** Persistence error for
temperature is strongly **bimodal** over the day — 0.68 °C at 04 UTC, a
**2.14 °C peak at 09 UTC** (morning warm-up), a 0.96 °C dip at 14 UTC (midday
plateau), a second **1.71 °C peak at 18 UTC** (evening cooling). A **3.14x**
swing, tracking the two diurnal transitions. Humidity matches in shape (2.40x);
pressure barely varies (1.53x); wind peaks once, in the afternoon.

**v27 flattens it**: 0.60-0.81 °C, a **1.36x** swing against persistence's
3.14x, with the LSTM between at 1.62x. Skill is correspondingly
hour-dependent — **+0.63 at 09 UTC** against **+0.13 at 04 UTC**; the pooled
+0.64 averages a fivefold range.

**Seasonal modulation (computed for the baselines).** Persistence's diurnal
amplitude for temperature scales exactly as the physics predicts:

| season | persistence amplitude | peak | 24h-lag amplitude |
|---|---|---|---|
| DJF | 2.80x | 1.92 °C @ 10 UTC | 1.21x |
| MAM | 3.25x | 2.21 °C @ 09 UTC | 1.37x |
| **JJA** | **4.02x** | **2.69 °C @ 08 UTC** | 1.61x |
| SON | 3.49x | 2.29 °C @ 09 UTC | 1.16x |

Summer has the steepest diurnal cycle, so persistence fails hardest there
(4.02x, peaking at 08 UTC as the sun climbs); winter is mildest (2.80x, peak
delayed to 10 UTC with the later sunrise). Note the **night floor is almost
season-independent** (0.66-0.69 °C except DJF's 0.69 at 07 UTC): stable nights
are equally predictable year-round, and it is the daytime transitions that
carry the seasonal signal. Pressure's amplitude stays low in every season
(1.56-2.15x), confirming it has no meaningful diurnal cycle.

**What the model panels test.** If v27's amplitude stays near 1 in *every*
season — especially JJA, where persistence is worst — the diurnal cycle is
genuinely learned rather than absorbed as an annual average. If instead v27's
amplitude rises in JJA alongside persistence, it is tracking state rather than
physics, and the annual-average flatness was partly a cancellation between
seasons. That distinction is the point of the split, and the amplitude table
answers it directly.

**For the report.** Do not quote one global skill-vs-climatology number: it
averages stations where the 24h-lag wins ~80% of hours with stations where it
wins 0%, and the split is altitude-driven (rho = -0.64). Quote
skill-vs-persistence globally, note its hour and season dependence, and quote
skill-vs-24h-lag only for temperature/humidity at >=4h leads.

**Hypothesis (not established here).** The altitude gradient plausibly reflects
boundary-layer decoupling: valley stations are radiatively driven and repeat
day to day, whereas high alpine stations sit in the free atmosphere and are
synoptically driven. Testing that needs radiation or stability data this
dataset does not contain.